# Win-back theo phân khúc: dồn ngân sách vào đâu?

Head of CRM có ngân sách cho **một** chiến dịch win-back nửa cuối năm và hỏi: **"Gửi cho nhóm nào thì đáng tiền?"**.
Bảng RFM ở bài Intermediate 01 là ảnh chụp **mô tả**; bài này kiểm nó như một DA senior: chốt phân khúc tại một ngày
trong quá khứ (01/06/2011), xem **6 tháng sau đó** mỗi phân khúc thật sự làm gì, rồi dựng bảng kịch bản giá trị chiến
dịch với giả định ghi rõ, và viết memo.

Dữ liệu: Online Retail II (UCI, CC BY 4.0), cả hai sheet, bỏ phần trùng 01-09/12/2010, dòng hợp lệ theo định nghĩa
dùng chung. Gợi ý cửa sổ: RFM trên 12 tháng trước ngày chốt (01/06/2010-31/05/2011), kết quả trong
[01/06/2011, 10/12/2011). Bạn được chọn khác, miễn là ghi rõ trong memo.

**Cách làm bài**
- Hai hàm có test: `segment_outcomes`, `campaign_value`. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Phần còn lại là phân tích tự do trong các cell `explore` (khi chấm bị bỏ qua) và `MEMO.md`.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/advanced/01-segment-winback-memo -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Viết sẵn, không cần sửa: đọc Online Retail II (cache parquet) và lọc dòng hợp lệ theo định nghĩa dùng chung
import tempfile
import uuid
from pathlib import Path

CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache trong thư mục tạm, không ghi vào data/raw


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet Online Retail II. Lần đầu đọc xlsx (30-90 giây) rồi cache parquet theo mtime và cỡ tệp."""
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    xlsx = root / "data" / "raw" / "online-retail-ii" / "online_retail_II.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py online-retail-ii")
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet"
    if cache.exists():
        try:
            return pd.read_parquet(cache)
        except (OSError, ValueError):  # cache hỏng: đọc lại
            pass
    tx = pd.read_excel(xlsx, sheet_name=sheet)
    tx = tx.assign(Invoice=tx["Invoice"].astype(str), StockCode=tx["StockCode"].astype(str),
                   Description=tx["Description"].astype("string"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{uuid.uuid4().hex}.tmp")
    tx.to_parquet(tmp, index=False)
    tmp.replace(cache)
    return tx


def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Bỏ hoá đơn huỷ (C...), Quantity <= 0, Price <= 0; thêm revenue = Quantity * Price."""
    s = tx.assign(Invoice=tx["Invoice"].astype(str))
    s = s[~s.Invoice.str.startswith("C") & (s.Quantity > 0) & (s.Price > 0)].copy()
    s["revenue"] = s.Quantity * s.Price
    return s

## 1. Phân khúc làm gì sau ngày chốt

`segment_outcomes(sales, segments, start, end) -> DataFrame`: `segments` là Series index mã khách, giá trị tên phân
khúc (mọi khách bạn xét). Chỉ dùng dòng có `Customer ID` nằm trong `segments` và `start <= InvoiceDate < end`. Index
phân khúc, cột theo thứ tự:

- `customers`: số khách của phân khúc (kể cả khách không mua gì trong cửa sổ);
- `active_share`: tỉ lệ khách có ít nhất một dòng trong cửa sổ;
- `revenue`: tổng `revenue` trong cửa sổ;
- `revenue_per_customer`: `revenue / customers`;
- `revenue_share`: tỉ trọng doanh thu của phân khúc trong tổng các phân khúc.

Sắp theo `revenue` giảm dần.

## 2. Bảng kịch bản giá trị chiến dịch

`campaign_value(outcomes, uplift, margin, cost_per_customer) -> DataFrame`: với mỗi phân khúc của `outcomes` (bảng
trên), giả sử chiến dịch làm thêm `uplift` (tỉ lệ khách, ví dụ 0.05 là thêm 5% khách) quay lại mua, mỗi khách thêm
mang lại doanh thu bằng **doanh thu trung bình của một khách active** của phân khúc (`revenue / (customers ×
active_share)`, 0 nếu không ai active). `uplift` là một số (chung) hoặc Series theo phân khúc (phân khúc thiếu coi là
0). Cột theo thứ tự:

- `customers`;
- `extra_gross_profit` = `customers × uplift × doanh thu mỗi khách active × margin`;
- `cost` = `customers × cost_per_customer`;
- `net` = `extra_gross_profit - cost`.

Sắp theo `net` giảm dần. **Uplift, margin, cost là giả định**: chỉ thử nghiệm có nhóm đối chứng mới đo được uplift thật.

In [ ]:
def segment_outcomes(sales: pd.DataFrame, segments: pd.Series, start, end) -> pd.DataFrame:
    """customers, active_share, revenue, revenue_per_customer, revenue_share theo phân khúc, trong [start, end)."""
    # TODO
    raise NotImplementedError


def campaign_value(outcomes: pd.DataFrame, uplift, margin: float, cost_per_customer: float) -> pd.DataFrame:
    """customers, extra_gross_profit, cost, net theo phân khúc; uplift là số hoặc Series theo phân khúc."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: dán rfm_table, score_rfm, SEGMENTS, assign_segment từ bài Intermediate 01 (hoặc tự định nghĩa phân khúc)
# TODO: phân khúc tại ngày chốt 01/06/2011 trên 12 tháng trước đó, rồi gọi segment_outcomes cho 01/06-10/12/2011

In [ ]:
# explore: phân tích tự do. Gợi ý: chart tỉ lệ quay lại theo phân khúc, doanh thu mỗi khách, tách at_risk theo M,
# kiểm khách / hoá đơn lớn nhất của từng phân khúc, bảng kịch bản campaign_value với uplift khác nhau theo phân khúc

## Viết memo

Chọn tối đa ba con số và một bảng cho Head of CRM. Viết `starter/MEMO.md` theo khung có sẵn (250-700 từ, dòng đầu
là headline có con số, ghi rõ chữ "giả định" ở mọi số không đến từ dữ liệu).